# 05 — Baseline ML Models

Phase 5 of the Indian Equity AI project plan.

Target: `next_day_direction` (1 = tomorrow's close > today's close).
Progression: naive base-rate baseline -> Logistic Regression -> Random Forest -> XGBoost -> LightGBM,
evaluated with a strict chronological (no-shuffle) train/validation/test split.

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
from lightgbm import LGBMClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier

from src.models.baseline import naive_predict, naive_predict_proba
from src.models.dataset import FEATURE_COLUMNS, TARGET_COLUMN, build_ml_dataset, chronological_split
from src.models.evaluate import classification_metrics

DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
RANDOM_STATE = 42

## Step 1: Build the ML dataset

In [2]:
technical = pd.read_parquet(DATA_PROCESSED / "technical_features.parquet")
dataset = build_ml_dataset(technical)

print("Rows:", len(dataset), "| Symbols:", dataset["symbol"].nunique())
print("Date range:", dataset["date"].min().date(), "to", dataset["date"].max().date())
print("Base rate (fraction of up days):", dataset[TARGET_COLUMN].mean().round(4))
dataset[FEATURE_COLUMNS + [TARGET_COLUMN]].describe().T[["mean", "std", "min", "max"]]

Rows: 51999 | Symbols: 50
Date range: 2022-07-18 to 2026-09-25
Base rate (fraction of up days): 0.5038


,mean,std,min,max
return_1d,0.000540,0.015855,-0.401513,0.200394
return_5d,0.002727,0.035298,-0.538131,0.661416
return_20d,0.011485,0.069824,-0.592012,0.461583
return_60d,0.034182,0.122146,-0.695730,1.091705
volatility_10d,0.014197,0.007150,0.002329,0.164903
volatility_20d,0.014639,0.006190,0.003449,0.117319
volatility_30d,0.014845,0.005745,0.004877,0.113123
volatility_60d,0.015191,0.005124,0.006354,0.082699
price_to_sma20,0.004455,0.038542,-0.529475,0.343843
price_to_sma50,0.012029,0.062286,-0.587857,0.407330


## Step 2: Chronological train / validation / test split

In [3]:
train, val, test = chronological_split(dataset, train_frac=0.7, val_frac=0.15)

for name, split in [("train", train), ("val", val), ("test", test)]:
    print(f"{name:5s}: {len(split):5d} rows | {split['date'].min().date()} to {split['date'].max().date()} "
          f"| base rate {split[TARGET_COLUMN].mean():.4f}")

X_train, y_train = train[FEATURE_COLUMNS], train[TARGET_COLUMN]
X_val, y_val = val[FEATURE_COLUMNS], val[TARGET_COLUMN]
X_test, y_test = test[FEATURE_COLUMNS], test[TARGET_COLUMN]

train: 36400 rows | 2022-07-18 to 2025-06-30 | base rate 0.5140
val  :  7800 rows | 2025-07-01 to 2026-02-12 | base rate 0.4891
test :  7799 rows | 2026-02-13 to 2026-09-25 | base rate 0.4712


## Step 3: Naive baseline (predict the training base rate)

In [4]:
results = {}

naive_val_proba = naive_predict_proba(y_train, n=len(y_val))
naive_val_pred = naive_predict(y_train, n=len(y_val))
results["naive_baseline"] = classification_metrics(y_val.values, naive_val_pred, naive_val_proba)
results["naive_baseline"]

{'accuracy': 0.4891025641025641,
 'precision': np.float64(0.4891025641025641),
 'recall': np.float64(1.0),
 'f1': np.float64(0.6569091691777873),
 'roc_auc': np.float64(0.5),
 'log_loss': 0.6941453731421802,
 'brier_score': np.float64(0.25049894135168854)}

## Step 4: Logistic Regression

In [5]:
scaler = StandardScaler().fit(X_train)
X_train_scaled = scaler.transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

logreg = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
logreg.fit(X_train_scaled, y_train)

logreg_val_proba = logreg.predict_proba(X_val_scaled)[:, 1]
logreg_val_pred = logreg.predict(X_val_scaled)
results["logistic_regression"] = classification_metrics(y_val.values, logreg_val_pred, logreg_val_proba)
results["logistic_regression"]

D:\Practise\Stock Market Analysis\venv\Lib\site-packages\sklearn\linear_model\_logistic.py:455: OptimizeWarning: Unknown solver options: iprint
  opt_res = optimize.minimize(


{'accuracy': 0.48833333333333334,
 'precision': np.float64(0.4847698165455175),
 'recall': np.float64(0.7342070773263434),
 'f1': np.float64(0.583967476284791),
 'roc_auc': np.float64(0.49552512616940003),
 'log_loss': 0.6957129575539104,
 'brier_score': np.float64(0.25116436951836424)}

## Step 5: Random Forest

In [6]:
rf = RandomForestClassifier(
    n_estimators=300, max_depth=6, min_samples_leaf=20, random_state=RANDOM_STATE, n_jobs=-1
)
rf.fit(X_train, y_train)

rf_val_proba = rf.predict_proba(X_val)[:, 1]
rf_val_pred = rf.predict(X_val)
results["random_forest"] = classification_metrics(y_val.values, rf_val_pred, rf_val_proba)
results["random_forest"]

{'accuracy': 0.4938461538461538,
 'precision': np.float64(0.49034412661536225),
 'recall': np.float64(0.8851900393184797),
 'f1': np.float64(0.631096991216595),
 'roc_auc': np.float64(0.4964351573972515),
 'log_loss': 0.6952547057892556,
 'brier_score': np.float64(0.2510490709775282)}

## Step 6: XGBoost

In [7]:
xgb = XGBClassifier(
    n_estimators=300,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric="logloss",
    random_state=RANDOM_STATE,
)
xgb.fit(X_train, y_train)

xgb_val_proba = xgb.predict_proba(X_val)[:, 1]
xgb_val_pred = xgb.predict(X_val)
results["xgboost"] = classification_metrics(y_val.values, xgb_val_pred, xgb_val_proba)
results["xgboost"]

{'accuracy': 0.49538461538461537,
 'precision': np.float64(0.4885525070955535),
 'recall': np.float64(0.6768020969855832),
 'f1': np.float64(0.5674725274725275),
 'roc_auc': np.float64(0.500550984935316),
 'log_loss': 0.6997651219137595,
 'brier_score': np.float64(0.25322577712073135)}

## Step 7: LightGBM

In [8]:
lgbm = LGBMClassifier(
    n_estimators=300,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=RANDOM_STATE,
    verbose=-1,
)
lgbm.fit(X_train, y_train)

lgbm_val_proba = lgbm.predict_proba(X_val)[:, 1]
lgbm_val_pred = lgbm.predict(X_val)
results["lightgbm"] = classification_metrics(y_val.values, lgbm_val_pred, lgbm_val_proba)
results["lightgbm"]

{'accuracy': 0.48923076923076925,
 'precision': np.float64(0.48447547308469596),
 'recall': np.float64(0.691218872870249),
 'f1': np.float64(0.5696694750486067),
 'roc_auc': np.float64(0.49490191757754753),
 'log_loss': 0.700547399669638,
 'brier_score': np.float64(0.25349519908403895)}

## Step 8: Compare all models on the validation set

In [9]:
comparison = pd.DataFrame(results).T
comparison.sort_values("roc_auc", ascending=False)

,accuracy,precision,recall,f1,roc_auc,log_loss,brier_score
xgboost,0.495385,0.488553,0.676802,0.567473,0.500551,0.699765,0.253226
naive_baseline,0.489103,0.489103,1.000000,0.656909,0.500000,0.694145,0.250499
random_forest,0.493846,0.490344,0.885190,0.631097,0.496435,0.695255,0.251049
logistic_regression,0.488333,0.484770,0.734207,0.583967,0.495525,0.695713,0.251164
lightgbm,0.489231,0.484475,0.691219,0.569669,0.494902,0.700547,0.253495


## Step 9: Evaluate the best validation model on the held-out test set

In [10]:
best_model_name = comparison["roc_auc"].idxmax()
print("Best model on validation (by ROC-AUC):", best_model_name)

models = {"logistic_regression": logreg, "random_forest": rf, "xgboost": xgb, "lightgbm": lgbm}
best_model = models.get(best_model_name)

if best_model_name == "logistic_regression":
    test_proba = best_model.predict_proba(X_test_scaled)[:, 1]
    test_pred = best_model.predict(X_test_scaled)
elif best_model is not None:
    test_proba = best_model.predict_proba(X_test)[:, 1]
    test_pred = best_model.predict(X_test)
else:
    test_proba = naive_predict_proba(y_train, n=len(y_test))
    test_pred = naive_predict(y_train, n=len(y_test))

test_metrics = classification_metrics(y_test.values, test_pred, test_proba)
test_metrics

Best model on validation (by ROC-AUC): xgboost


{'accuracy': 0.4931401461725862,
 'precision': np.float64(0.4737834779328555),
 'recall': np.float64(0.6835374149659864),
 'f1': np.float64(0.5596524451375738),
 'roc_auc': np.float64(0.5068882334699156),
 'log_loss': 0.700993050449037,
 'brier_score': np.float64(0.2537979837385257)}

## Step 10: Feature importance (Random Forest and XGBoost)

In [11]:
importance = pd.DataFrame(
    {
        "random_forest": rf.feature_importances_,
        "xgboost": xgb.feature_importances_,
    },
    index=FEATURE_COLUMNS,
)
importance["avg_rank"] = importance.rank(ascending=False).mean(axis=1)
importance.sort_values("avg_rank").head(10)

,random_forest,xgboost,avg_rank
price_to_sma20,0.044734,0.044289,3.0
atr_pct,0.060719,0.043157,4.0
volatility_10d,0.058615,0.042924,5.0
roc_12,0.041630,0.044046,6.0
volume_ratio,0.068853,0.042141,6.0
technical_score,0.042048,0.043664,6.5
sma50_to_sma200,0.043243,0.042160,8.5
return_1d,0.051361,0.041820,8.5
volatility_20d,0.040935,0.042827,10.0
price_to_sma200,0.042441,0.041924,10.0


In [12]:
comparison.to_csv(DATA_PROCESSED / "baseline_model_validation_metrics.csv")
pd.Series(test_metrics, name=best_model_name).to_csv(
    DATA_PROCESSED / "baseline_best_model_test_metrics.csv"
)
importance.to_csv(DATA_PROCESSED / "baseline_feature_importance.csv")
print("Saved validation comparison, test metrics, and feature importance to data/processed/")

Saved validation comparison, test metrics, and feature importance to data/processed/


## Reflection

**What worked:**
- The chronological split + NaN-safe target labeling pipeline runs cleanly end to end, pooling
  all 5 stocks into a single cross-sectional training set (as opposed to training one model
  per stock), which gives the tree models a meaningfully larger and more diverse training set.
- The naive base-rate baseline behaves exactly as expected -- ROC-AUC ~0.5, confirming the
  evaluation pipeline itself is correct before trusting any "skillful" model's numbers.
- All four models fit and produce well-formed probability outputs (log loss and Brier score are
  finite, ROC-AUC computable), so the infrastructure (dataset -> split -> train -> evaluate) is solid.

**What didn't (or is expected to be weak):**
- Next-day directional prediction from price/technical features alone is a famously hard,
  low-signal problem (weak-form market efficiency). Do not expect ROC-AUC much above ~0.52-0.55
  here -- if a model reports something dramatically higher, that is a signal to suspect leakage
  before celebrating.
- Only 5 stocks and one non-overlapping chronological split means the validation/test sets are
  small; metric differences between models may not be statistically meaningful yet. Walk-forward
  validation (Phase 9) will give a much more reliable read.
- This baseline intentionally excludes fundamentals, news/sentiment, and macro data (per the
  project plan's phased design) -- any predictive edge found here is technical-only.

**Which features mattered:** see the feature-importance table above (Step 10) -- inspect it for
the models actually trained in this run before drawing conclusions.

**What's next:** Phase 6 (news/sentiment) and Phase 7 (multimodal ensemble) are the next places
genuinely new information could come from, rather than further hyperparameter-tuning this
technical-only baseline.